# L03 · PagedAttention
见 `lectures/03-paged-attention.md`

In [ ]:
import sys; sys.path.insert(0,'../src')
import torch
from paged_kv import PagedKvPool, BlockTable, utilization

pool = PagedKvPool(n_blocks=128, block_size=16, n_kv_heads=2, head_dim=32, n_layers=1)
tables = [BlockTable(pool) for _ in range(4)]
for t, n in zip(tables, [40,17,95,8]):
    for i in range(n):
        t.append_token(0, torch.randn(2,32,dtype=torch.float16), torch.randn(2,32,dtype=torch.float16))
print(f'util={utilization(tables)*100:.1f}%')
print(f'free={pool.n_free()}/{pool.n_blocks}')

In [ ]:
# Fork demo (COW): no extra physical blocks
free_before = pool.n_free()
child = tables[0].fork()
print(f'parent blocks: {tables[0].block_ids}')
print(f'child  blocks: {child.block_ids}')
print(f'free unchanged: {pool.n_free()==free_before}')